## Building A Chatbot
In this video We'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for:

- Conversational RAG: Enable a chatbot experience over an external source of data
- Agents: Build a chatbot that can take actions

This video tutorial will cover the basics which will be helpful for those two more advanced topics.

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()
groq_api_key=os.getenv("GROQ_API_KEY")


In [6]:
from langchain_groq import ChatGroq
model=ChatGroq(model="llama-3.1-8b-instant",groq_api_key=groq_api_key)
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001CC28D29990>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001CC28D2B520>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [ ]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(
    content='Hi, My name is Bilal and I am a chief AI Engineer at a company called Agentic AI')
])

AIMessage(content="Nice to meet you, Bilal. Congratulations on being the Chief AI Engineer at Agentic AI. That sounds like a fascinating role. I'd be happy to learn more about your work and the company. What areas of AI are you and your team focusing on, and what are some of the exciting projects you're currently working on?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 68, 'prompt_tokens': 56, 'total_tokens': 124, 'completion_time': 0.111430786, 'completion_tokens_details': None, 'prompt_time': 0.002850074, 'prompt_tokens_details': None, 'queue_time': 0.018489303, 'total_time': 0.11428086}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_03e8423237', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e0b5f-307a-7823-b6f9-df03a6e21717-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 56, 'output_tokens': 68, 'total_tokens': 124})

In [8]:
from langchain_core.messages import HumanMessage, AIMessage

model.invoke(
             [
                HumanMessage( content='Hi, My name is Bilal and I am a chief AI Engineer at a company called Agentic AI'),
                AIMessage(content="Nice to meet you, Bilal. Congratulations on being the Chief AI Engineer at Agentic AI. That sounds like a fascinating role. I'd be happy to learn more about your work and the company. What areas of AI are you and your team focusing on, and what are some of the exciting projects you're currently working on?"),
                HumanMessage(content="Hey What is my name and what do I do?")
             ]
)








AIMessage(content='Your name is Bilal, and you are the Chief AI Engineer at Agentic AI.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 144, 'total_tokens': 163, 'completion_time': 0.038799799, 'completion_tokens_details': None, 'prompt_time': 0.019008129, 'prompt_tokens_details': None, 'queue_time': 0.046183786, 'total_time': 0.057807928}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_d9492c3c54', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e0b62-4223-7481-b50d-b684f4e91355-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 144, 'output_tokens': 19, 'total_tokens': 163})

## Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [13]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:id)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory(session_id=session_id)
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model, get_session_history)


d:\2026-courses\agenticai\venv\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [14]:
config = {
    "configurable":{
        "session_id":"chat1"
    }
}

In [17]:
response = with_message_history.invoke(
    [HumanMessage(content='Hi, My name is Bilal and I am a chief AI Engineer at a company called Agentic AI'),],
    config=config
)

In [18]:
response.content

'Hello Bilal. What specific areas of Artificial Intelligence is Agentic AI focusing on, and what are some of the exciting projects that your team is currently working on?'

In [19]:
with_message_history.invoke(
    [HumanMessage(content="Hey What is my name and what do I do?")],
    config=config
)

AIMessage(content="Your name is Bilal, and you're the Chief AI Engineer at a company called Agentic AI.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 204, 'total_tokens': 226, 'completion_time': 0.020699769, 'completion_tokens_details': None, 'prompt_time': 0.012013673, 'prompt_tokens_details': None, 'queue_time': 0.0186509, 'total_time': 0.032713442}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_6a1eabf260', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e0b6c-3743-70d3-950f-9bdc813bd818-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 204, 'output_tokens': 22, 'total_tokens': 226})

In [21]:
## Change the session id
config1 = {
    "configurable":{
        "session_id":"chat2"
    }
}

response= with_message_history.invoke(
    [HumanMessage(content="Hey What is my name and what do I do?")],
    config=config1
)

In [22]:
response.content

"As we've established earlier, I don't have any information about your personal identity or profession. However, I can suggest a fun way to play this out.\n\nLet's imagine a different scenario. I can suggest a few options for your name and profession, and you can choose which one you like best. Would you like to do that?"